# 03 — Evaluation: Fine-Tuned Mistral 7B vs GPT-4o-mini

**Goal:** Measure field extraction accuracy of our fine-tuned model and compare against GPT-4o-mini (Azure) baseline on the same eval set.

**Metrics:**
- Per-field accuracy (exact match for numbers/dates, fuzzy match for text)
- Line item matching score
- JSON parse success rate
- Overall accuracy with improvement percentage

## 1. Setup

In [ ]:
!pip install -q transformers peft bitsandbytes openai rapidfuzz wandb python-dotenv

In [ ]:
import os, sys, json
from dotenv import load_dotenv

sys.path.insert(0, '..')
load_dotenv()

## 2. Load Eval Data

In [ ]:
from src.data.format import load_jsonl
from src.data.schema import Invoice

eval_data = load_jsonl('../data/eval.jsonl')
print(f'Eval samples: {len(eval_data)}')

In [ ]:
# Parse gold labels
gold_invoices = [Invoice.model_validate_json(ex['output']) for ex in eval_data]
print(f'Gold invoices parsed: {len(gold_invoices)}')

Preview one eval example:

In [ ]:
print('Input (first 200 chars):')
print(eval_data[0]['input'][:200])
print(f'\nGold output:')
print(gold_invoices[0].model_dump_json(indent=2)[:300])

---
## 3. Run Fine-Tuned Model

Load the base model + LoRA adapter and run inference on all eval samples.

In [ ]:
from src.evaluation.evaluate import load_finetuned_model, run_finetuned_inference

model, tokenizer = load_finetuned_model(
    base_model='mistralai/Mistral-7B-v0.3',
    adapter_path='../outputs/final_adapter',
)

In [ ]:
ft_predictions = run_finetuned_inference(model, tokenizer, eval_data)

ft_parsed = sum(1 for p in ft_predictions if p is not None)
print(f'Parsed: {ft_parsed}/{len(ft_predictions)} ({ft_parsed/len(ft_predictions)*100:.1f}%)')

---
## 4. Run GPT-4o-mini Baseline

Run the same eval samples through GPT-4o-mini (Azure) with zero-shot prompting — no fine-tuning, just the schema in the system prompt.

In [ ]:
from src.evaluation.baseline import run_baseline

baseline_predictions = run_baseline(eval_data)

bl_parsed = sum(1 for p in baseline_predictions if p is not None)
print(f'Parsed: {bl_parsed}/{len(baseline_predictions)} ({bl_parsed/len(baseline_predictions)*100:.1f}%)' )

---
## 5. Compute Metrics

In [ ]:
from src.evaluation.evaluate import aggregate_metrics

ft_metrics = aggregate_metrics(ft_predictions, gold_invoices)
baseline_metrics = aggregate_metrics(baseline_predictions, gold_invoices)

In [ ]:
print(f"{'Metric':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12}")
print('-' * 52)
print(f"{'Overall Accuracy':<25} {ft_metrics['overall_accuracy']:>11.1%} {baseline_metrics['overall_accuracy']:>11.1%}")
print(f"{'JSON Parse Rate':<25} {ft_metrics['json_parse_success_rate']:>11.1%} {baseline_metrics['json_parse_success_rate']:>11.1%}")
print(f"{'Line Item Score':<25} {ft_metrics['line_item_score']:>11.1%} {baseline_metrics['line_item_score']:>11.1%}")

### Per-Field Accuracy

In [ ]:
print(f"{'Field':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12} {'Improvement':>12}")
print('-' * 64)

all_fields = sorted(set(list(ft_metrics['per_field'].keys()) + list(baseline_metrics['per_field'].keys())))
for field in all_fields:
    ft_val = ft_metrics['per_field'].get(field, 0)
    bl_val = baseline_metrics['per_field'].get(field, 0)
    imp = ((ft_val - bl_val) / bl_val * 100) if bl_val > 0 else 0
    print(f'{field:<25} {ft_val:>11.1%} {bl_val:>11.1%} {imp:>+11.1f}%')

---
## 6. Generate Report

In [ ]:
from src.evaluation.evaluate import generate_report

report = generate_report(ft_metrics, baseline_metrics)

os.makedirs('../reports', exist_ok=True)
with open('../reports/evaluation_report.md', 'w') as f:
    f.write(report)

print('Report saved to reports/evaluation_report.md')

In [ ]:
print(report)

---
## 7. Log to W&B

In [ ]:
import wandb

wandb.init(project='invoice-extraction-finetune', job_type='evaluation')

In [ ]:
wandb.log({
    'ft_overall_accuracy': ft_metrics['overall_accuracy'],
    'baseline_overall_accuracy': baseline_metrics['overall_accuracy'],
    'ft_parse_rate': ft_metrics['json_parse_success_rate'],
    'baseline_parse_rate': baseline_metrics['json_parse_success_rate'],
    'improvement_pct': (
        (ft_metrics['overall_accuracy'] - baseline_metrics['overall_accuracy'])
        / baseline_metrics['overall_accuracy'] * 100
    ) if baseline_metrics['overall_accuracy'] > 0 else 0,
})

In [ ]:
for field in ft_metrics['per_field']:
    wandb.log({
        f'ft_{field}': ft_metrics['per_field'].get(field, 0),
        f'baseline_{field}': baseline_metrics['per_field'].get(field, 0),
    })

wandb.finish()
print('Metrics logged to W&B')

---
## 8. Conclusion

Update the improvement percentage in the resume bullet:

> "Fine-tuned Mistral 7B on invoice extraction dataset using QLoRA; achieved **X% improvement** in field extraction accuracy over GPT-4o-mini baseline, measured across N-sample eval set."